# DICE ITC 03: Design-Space Exploration and Complexity

This notebook isolates the DSE figures and the projected score-stage accelerator complexity estimate.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import sys
import tempfile
import types

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
CFG_LABEL = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}


def _cfg_labels(values: pd.Series) -> list[str]:
    return [CFG_LABEL.get(str(v), str(v)) for v in values]


def render_tier_correlation_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    tier_case = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    tier_final = tier_case[tier_case["config"] == "tier0_tier1_tier2"].copy()
    tier_cols = ["tier0_share", "tier1_alt_share", "tier2_share"]

    tier_corr = tier_final[tier_cols].corr().round(4)
    tier_corr.to_csv(paper_full / "tier_share_correlation.csv")

    stressor_tier = tier_final.groupby("stressor", sort=False)[tier_cols].mean().reset_index()
    stressor_tier.to_csv(paper_full / "stressor_tier_share_summary.csv", index=False)

    tier_final["ternary_x"] = tier_final["tier1_alt_share"] + 0.5 * tier_final["tier2_share"]
    tier_final["ternary_y"] = (np.sqrt(3.0) / 2.0) * tier_final["tier2_share"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    im = axes[0].imshow(tier_corr.values, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_xticks(range(3), ["Tier-0", "Tier-1", "Tier-2"], rotation=30, ha="right")
    axes[0].set_yticks(range(3), ["Tier-0", "Tier-1", "Tier-2"])
    axes[0].set_title("Tier-share correlation")
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f"{tier_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(stressor_tier))
    axes[1].bar(x, stressor_tier["tier0_share"], label="Tier-0")
    axes[1].bar(x, stressor_tier["tier1_alt_share"], bottom=stressor_tier["tier0_share"], label="Tier-1")
    axes[1].bar(
        x,
        stressor_tier["tier2_share"],
        bottom=stressor_tier["tier0_share"] + stressor_tier["tier1_alt_share"],
        label="Tier-2",
    )
    axes[1].set_xticks(x, stressor_tier["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Mean tier evidence by stressor")
    axes[1].legend(loc="upper right")

    triangle = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [0.5, np.sqrt(3.0) / 2.0],
            [0.0, 0.0],
        ]
    )
    axes[2].plot(triangle[:, 0], triangle[:, 1], color="black")
    for stressor, d in tier_final.groupby("stressor", sort=False):
        axes[2].scatter(d["ternary_x"], d["ternary_y"], s=36, alpha=0.8, label=stressor)
    axes[2].text(-0.04, -0.03, "Tier-0")
    axes[2].text(1.01, -0.03, "Tier-1")
    axes[2].text(0.46, np.sqrt(3.0) / 2.0 + 0.03, "Tier-2")
    axes[2].set_title("Per-case tier composition")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
    axes[2].legend(loc="upper right", fontsize=7)

    fig.tight_layout()
    png = paper_fig / "fig_tier_correlation_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_corr, stressor_tier, png


def render_bootstrap_confidence(
    case_pred: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
    samples: int = 1000,
    seed: int = 0,
) -> tuple[pd.DataFrame, Path]:
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for cfg, d in case_pred.groupby("config", sort=False):
        stats: list[dict[str, float]] = []
        for _ in range(samples):
            sample = d.sample(n=len(d), replace=True, random_state=int(rng.integers(1 << 32)))
            if sample["label"].nunique() < 2:
                continue
            benign = sample[sample["label"] == 0]
            anomaly = sample[sample["label"] == 1]
            stats.append(
                {
                    "roc_auc_wc": roc_auc_score(sample["label"], sample["run_score_wc"]),
                    "pr_auc_wc": average_precision_score(sample["label"], sample["run_score_wc"]),
                    "benign_run_false_alarm_rate": benign["run_alert"].mean(),
                    "anomaly_run_detection_rate": anomaly["run_alert"].mean(),
                    "median_time_to_detect_s": anomaly.loc[
                        anomaly["run_alert"] == 1, "time_to_detect_s"
                    ].median(),
                }
            )

        boot = pd.DataFrame(stats)
        rows.append(
            {
                "config": cfg,
                "roc_auc_wc_lo": boot["roc_auc_wc"].quantile(0.025),
                "roc_auc_wc_hi": boot["roc_auc_wc"].quantile(0.975),
                "pr_auc_wc_lo": boot["pr_auc_wc"].quantile(0.025),
                "pr_auc_wc_hi": boot["pr_auc_wc"].quantile(0.975),
                "fpr_lo": boot["benign_run_false_alarm_rate"].quantile(0.025),
                "fpr_hi": boot["benign_run_false_alarm_rate"].quantile(0.975),
                "detect_lo": boot["anomaly_run_detection_rate"].quantile(0.025),
                "detect_hi": boot["anomaly_run_detection_rate"].quantile(0.975),
                "ttd_lo": boot["median_time_to_detect_s"].quantile(0.025),
                "ttd_hi": boot["median_time_to_detect_s"].quantile(0.975),
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(paper_full / "bootstrap_confidence_intervals.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    x = np.arange(len(out))

    pr_mid = (out["pr_auc_wc_lo"] + out["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - out["pr_auc_wc_lo"], out["pr_auc_wc_hi"] - pr_mid])
    axes[0].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4)
    axes[0].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[0].set_title("Bootstrap AUC-PR CI")

    fpr_mid = (out["fpr_lo"] + out["fpr_hi"]) / 2.0
    fpr_err = np.vstack([fpr_mid - out["fpr_lo"], out["fpr_hi"] - fpr_mid])
    axes[1].errorbar(x, fpr_mid, yerr=fpr_err, fmt="o", capsize=4)
    axes[1].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[1].set_title("Bootstrap benign-FPR CI")

    ttd_mid = (out["ttd_lo"] + out["ttd_hi"]) / 2.0
    ttd_err = np.vstack([ttd_mid - out["ttd_lo"], out["ttd_hi"] - ttd_mid])
    axes[2].errorbar(x, ttd_mid, yerr=ttd_err, fmt="o", capsize=4)
    axes[2].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[2].set_title("Bootstrap time-to-detect CI")

    fig.tight_layout()
    png = paper_fig / "fig_bootstrap_confidence_intervals.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out, png


def export_llm_case_cards(
    out_full: Path,
    appendix_full: Path,
) -> pd.DataFrame:
    case_diag = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    cards = case_diag[case_diag["config"] == "tier0_tier1_tier2"].copy()
    keep_cols = [
        "case_id",
        "workload",
        "stressor",
        "dominant_tier",
        "dominant_mechanism",
        "tier0_share",
        "tier1_alt_share",
        "tier2_share",
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
        "top_feature_1",
        "top_feature_score_1",
        "top_feature_2",
        "top_feature_score_2",
        "top_feature_3",
        "top_feature_score_3",
        "top_feature_4",
        "top_feature_score_4",
        "top_feature_5",
        "top_feature_score_5",
        "top_mechanism_1",
        "top_mechanism_score_1",
        "top_mechanism_2",
        "top_mechanism_score_2",
        "top_mechanism_3",
        "top_mechanism_score_3",
    ]
    cards = cards[[c for c in keep_cols if c in cards.columns]].copy()

    def _case_card_json(row: pd.Series) -> str:
        payload = {
            "case_id": row.get("case_id"),
            "workload": row.get("workload"),
            "stressor": row.get("stressor"),
            "dominant_tier": row.get("dominant_tier"),
            "dominant_mechanism": row.get("dominant_mechanism"),
            "tier_share": {
                "tier0": row.get("tier0_share"),
                "tier1": row.get("tier1_alt_share"),
                "tier2": row.get("tier2_share"),
            },
            "mechanism_share": {
                "compute": row.get("compute_share"),
                "memory_io": row.get("memory_io_share"),
                "thermal_power": row.get("thermal_power_share"),
                "scheduler_runtime": row.get("scheduler_runtime_share"),
                "platform_pressure": row.get("platform_pressure_share"),
            },
            "top_features": [
                {"name": row.get(f"top_feature_{i}"), "score": row.get(f"top_feature_score_{i}")}
                for i in range(1, 6)
                if pd.notna(row.get(f"top_feature_{i}"))
            ],
            "top_mechanisms": [
                {"name": row.get(f"top_mechanism_{i}"), "score": row.get(f"top_mechanism_score_{i}")}
                for i in range(1, 4)
                if pd.notna(row.get(f"top_mechanism_{i}"))
            ],
        }
        return json.dumps(payload, sort_keys=True)

    cards["diagnostic_case_card_json"] = cards.apply(_case_card_json, axis=1)
    cards["reviewer_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "You are preparing a grounded DICE diagnostic note for a silicon-reliability reviewer. "
            "Use only the supplied case card. Do not invent missing evidence. "
            "Explain the dominant tier, dominant mechanism, and the top residual cues in plain English.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["triage_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Write a compact triage report with five fields: "
            "severity, likely subsystem, evidence summary, two follow-up measurements, and confidence. "
            "If the evidence is weak, say so directly.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["followup_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Recommend up to three next diagnostic steps. "
            "Each step must cite the specific feature or mechanism that motivated it.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards.to_csv(appendix_full / "llm_case_cards.csv", index=False)
    return cards


def export_llm_diagnostic_model_catalog(appendix_full: Path) -> pd.DataFrame:
    models = pd.DataFrame(
        [
            {
                "model_id": "Qwen/Qwen2.5-7B-Instruct",
                "deployment_role": "primary DICE baseline",
                "priority_rank": 1,
                "params_billions": 7.61,
                "context_tokens": 131072,
                "strengths": "Strong instruction following, structured output behavior, and long-context support.",
                "best_for_dice": "Primary grounded reviewer summaries and structured incident reports from exported case cards.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct",
            },
            {
                "model_id": "microsoft/Phi-4-mini-instruct",
                "deployment_role": "lightweight comparison",
                "priority_rank": 2,
                "params_billions": 3.8,
                "context_tokens": 128000,
                "strengths": "Small footprint, strong reasoning density, and good fit for constrained local diagnostics.",
                "best_for_dice": "Fast first-pass case summaries and follow-up recommendations on a laptop or edge workstation.",
                "source_url": "https://huggingface.co/microsoft/Phi-4-mini-instruct",
            },
            {
                "model_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
                "deployment_role": "ecosystem baseline",
                "priority_rank": 3,
                "params_billions": 8.0,
                "context_tokens": 128000,
                "strengths": "Broad tooling support, stable chat behavior, and strong general-purpose instruction tuning.",
                "best_for_dice": "Fallback baseline when the deployment stack already supports Llama-family models.",
                "source_url": "https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct",
            },
            {
                "model_id": "Qwen/Qwen2.5-14B-Instruct",
                "deployment_role": "stronger offline review",
                "priority_rank": 4,
                "params_billions": 14.7,
                "context_tokens": 131072,
                "strengths": "Higher-capacity structured reasoning while remaining practical for offline workstation use.",
                "best_for_dice": "Second-pass failure analysis and richer postmortem summaries after the detector has already raised a case.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-14B-Instruct",
            },
        ]
    ).sort_values('priority_rank').reset_index(drop=True)
    models.to_csv(appendix_full / "llm_diagnostic_model_catalog.csv", index=False)
    return models


def export_llm_diagnostic_prompt_bundle(
    cards: pd.DataFrame,
    models: pd.DataFrame,
    appendix_full: Path,
) -> pd.DataFrame:
    system_prompt = (
        "You are a DICE diagnostic copilot. You may use only the structured DICE evidence supplied to you. "
        "Do not claim access to raw telemetry, hidden logs, or external knowledge about the run. "
        "If the evidence is incomplete, say that the conclusion is tentative."
    )
    rows = []
    for _, model in models.iterrows():
        for _, card in cards.iterrows():
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "triage",
                    "system_prompt": system_prompt,
                    "user_prompt": card["triage_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "reviewer_summary",
                    "system_prompt": system_prompt,
                    "user_prompt": card["reviewer_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "followup",
                    "system_prompt": system_prompt,
                    "user_prompt": card["followup_prompt"],
                }
            )

    bundle = pd.DataFrame(rows)
    bundle.to_csv(appendix_full / "llm_diagnostic_prompt_bundle.csv", index=False)
    with (appendix_full / "llm_diagnostic_prompt_bundle.jsonl").open("w") as f:
        for row in bundle.to_dict(orient="records"):
            f.write(json.dumps(row) + "\n")
    return bundle



TIER_ALIAS_MAP = {
    "tier0": ["tier0", "tier-0", "tier 0", "tier-0 evidence", "tier 0 evidence"],
    "tier1_alt": ["tier1", "tier-1", "tier 1", "tier1_alt", "tier-1 evidence", "tier 1 evidence"],
    "tier2": ["tier2", "tier-2", "tier 2", "tier-2 evidence", "tier 2 evidence"],
}

MECHANISM_ALIAS_MAP = {
    "compute": ["compute"],
    "memory_io": ["memory_io", "memory/io", "memory io"],
    "thermal_power": ["thermal_power", "thermal/power", "thermal power"],
    "scheduler_runtime": ["scheduler_runtime", "scheduler runtime"],
    "platform_pressure": ["platform_pressure", "platform pressure"],
}


def _slugify_model_id(model_id: str) -> str:
    return model_id.replace('/', '__').replace('-', '_').replace('.', '_')


def _normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace('_', ' ')
    text = text.replace(':', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def _feature_aliases(name: str | float | None) -> set[str]:
    if pd.isna(name):
        return set()
    name = str(name)
    aliases = {name.lower(), _normalize_text(name)}
    if ':' in name:
        tail = name.split(':', 1)[1]
        aliases.add(tail.lower())
        aliases.add(_normalize_text(tail))
    return {a for a in aliases if a}


def _contains_any(text: str, aliases: set[str] | list[str]) -> bool:
    norm = _normalize_text(text)
    return any(alias and _normalize_text(alias) in norm for alias in aliases)


def run_transformers_llm_batch(
    prompt_bundle: pd.DataFrame,
    appendix_full: Path,
    model_id: str,
    prompt_type: str = "triage",
    max_cases: int = 8,
    max_new_tokens: int = 320,
    temperature: float = 0.0,
) -> pd.DataFrame:
    if importlib.util.find_spec("transformers") is None or importlib.util.find_spec("torch") is None:
        raise RuntimeError("transformers and torch must be installed to run local model inference.")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    subset = prompt_bundle[
        (prompt_bundle["model_id"] == model_id) & (prompt_bundle["prompt_type"] == prompt_type)
    ].copy().head(max_cases)
    if subset.empty:
        raise ValueError(f"No prompts found for model_id={model_id!r} and prompt_type={prompt_type!r}.")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )

    rows = []
    for row in subset.itertuples(index=False):
        messages = [
            {"role": "system", "content": row.system_prompt},
            {"role": "user", "content": row.user_prompt},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            text = row.system_prompt + "\n\n" + row.user_prompt

        model_inputs = tokenizer([text], return_tensors="pt")
        model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=max(temperature, 1e-5),
        )
        new_ids = generated_ids[:, model_inputs["input_ids"].shape[1]:]
        response_text = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
        rows.append(
            {
                "model_id": model_id,
                "case_id": row.case_id,
                "prompt_type": prompt_type,
                "response_text": response_text,
            }
        )

    outputs = pd.DataFrame(rows)
    slug = _slugify_model_id(model_id)
    outputs.to_csv(appendix_full / f"llm_outputs_{slug}_{prompt_type}.csv", index=False)
    with (appendix_full / f"llm_outputs_{slug}_{prompt_type}.jsonl").open("w") as f:
        for rec in outputs.to_dict(orient="records"):
            f.write(json.dumps(rec) + "\n")
    return outputs


def evaluate_llm_grounding_outputs(
    llm_outputs: pd.DataFrame,
    llm_cards: pd.DataFrame,
    appendix_full: Path,
    stem: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    detail = llm_outputs.merge(llm_cards, on="case_id", how="left", suffixes=("", "_card")).copy()

    supported_tier_threshold = 0.05
    supported_mech_threshold = 0.10
    global_mechanisms = list(MECHANISM_ALIAS_MAP.keys())
    global_tiers = list(TIER_ALIAS_MAP.keys())

    rows = []
    for row in detail.itertuples(index=False):
        text = str(row.response_text)
        dominant_tier = getattr(row, "dominant_tier")
        dominant_mechanism = getattr(row, "dominant_mechanism")

        tier_aliases = set(TIER_ALIAS_MAP.get(str(dominant_tier), []))
        dominant_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(dominant_mechanism), [str(dominant_mechanism)]))
        top_feature_aliases = _feature_aliases(getattr(row, "top_feature_1", None))
        top_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(getattr(row, "top_mechanism_1", '')), [str(getattr(row, "top_mechanism_1", ''))]))

        mentions_dominant_tier = _contains_any(text, tier_aliases)
        mentions_dominant_mechanism = _contains_any(text, dominant_mech_aliases)
        mentions_top_feature_1 = _contains_any(text, top_feature_aliases)
        mentions_top_mechanism_1 = _contains_any(text, top_mech_aliases)

        supported_tiers = {
            "tier0": getattr(row, "tier0_share", 0.0),
            "tier1_alt": getattr(row, "tier1_alt_share", 0.0),
            "tier2": getattr(row, "tier2_share", 0.0),
        }
        unsupported_tier_mentions = any(
            _contains_any(text, TIER_ALIAS_MAP[t]) and supported_tiers.get(t, 0.0) < supported_tier_threshold
            for t in global_tiers
        )

        supported_mechs = {
            "compute": getattr(row, "compute_share", 0.0),
            "memory_io": getattr(row, "memory_io_share", 0.0),
            "thermal_power": getattr(row, "thermal_power_share", 0.0),
            "scheduler_runtime": getattr(row, "scheduler_runtime_share", 0.0),
            "platform_pressure": getattr(row, "platform_pressure_share", 0.0),
        }
        unsupported_mechanism_mentions = any(
            _contains_any(text, MECHANISM_ALIAS_MAP[m]) and supported_mechs.get(m, 0.0) < supported_mech_threshold
            for m in global_mechanisms
        )

        cue_coverage = np.mean([
            float(mentions_dominant_tier),
            float(mentions_dominant_mechanism),
            float(mentions_top_feature_1),
            float(mentions_top_mechanism_1),
        ])

        rows.append(
            {
                "model_id": getattr(row, "model_id"),
                "case_id": getattr(row, "case_id"),
                "prompt_type": getattr(row, "prompt_type"),
                "mentions_dominant_tier": mentions_dominant_tier,
                "mentions_dominant_mechanism": mentions_dominant_mechanism,
                "mentions_top_feature_1": mentions_top_feature_1,
                "mentions_top_mechanism_1": mentions_top_mechanism_1,
                "grounded_core": bool(mentions_dominant_tier and mentions_dominant_mechanism),
                "cue_coverage": cue_coverage,
                "unsupported_tier_mentions": unsupported_tier_mentions,
                "unsupported_mechanism_mentions": unsupported_mechanism_mentions,
                "hallucination_flag": bool(unsupported_tier_mentions or unsupported_mechanism_mentions),
            }
        )

    scored = pd.DataFrame(rows)
    summary = (
        scored.groupby(["model_id", "prompt_type"], sort=False)
        .agg(
            n_outputs=("case_id", "count"),
            grounded_core_rate=("grounded_core", "mean"),
            dominant_tier_rate=("mentions_dominant_tier", "mean"),
            dominant_mechanism_rate=("mentions_dominant_mechanism", "mean"),
            top_feature_1_rate=("mentions_top_feature_1", "mean"),
            top_mechanism_1_rate=("mentions_top_mechanism_1", "mean"),
            mean_cue_coverage=("cue_coverage", "mean"),
            hallucination_rate=("hallucination_flag", "mean"),
        )
        .reset_index()
    )

    summary.to_csv(appendix_full / f"llm_grounding_summary_{stem}.csv", index=False)
    scored.to_csv(appendix_full / f"llm_grounding_detail_{stem}.csv", index=False)
    return summary, scored


def render_paper_performance_stack(
    case_pred: pd.DataFrame,
    overall_full: pd.DataFrame,
    sequential: pd.DataFrame,
    reliability: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    summary = (
        overall_full[
            [
                "config",
                "roc_auc",
                "pr_auc",
                "roc_auc_wc",
                "pr_auc_wc",
                "median_nominal_score_wc",
                "median_anomaly_score_wc",
            ]
        ]
        .merge(
            sequential[
                [
                    "config",
                    "anomaly_detect_rate",
                    "median_time_to_detect_s",
                ]
            ],
            on="config",
        )
        .merge(
            reliability[
                [
                    "config",
                    "target_alpha",
                    "benign_block_false_alarm_rate",
                ]
            ],
            on="config",
        )
    )
    summary["config_label"] = _cfg_labels(summary["config"])
    summary["reliability_margin"] = summary["target_alpha"] - summary["benign_block_false_alarm_rate"]
    summary.to_csv(paper_full / "paper_performance_stack_summary.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    final = case_pred[case_pred["config"] == "tier0_tier1_tier2"].copy()
    benign = final[final["label"] == 0]["run_score_wc"].to_numpy(dtype=float)
    anomaly = final[final["label"] == 1]["run_score_wc"].to_numpy(dtype=float)
    bp = axes[0, 0].boxplot([benign, anomaly], labels=["Benign", "Anomaly"], patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#2563eb", "#dc2626"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[0, 0].set_title("A. Final-head score separation")
    axes[0, 0].set_ylabel("Workload-conditioned run score")

    base_color = "#94a3b8"
    wc_color = "#0f766e"
    for _, row in summary.iterrows():
        axes[0, 1].scatter(row["roc_auc"], row["pr_auc"], color=base_color, s=70)
        axes[0, 1].scatter(row["roc_auc_wc"], row["pr_auc_wc"], color=wc_color, s=90)
        axes[0, 1].annotate(
            row["config_label"],
            (row["roc_auc_wc"], row["pr_auc_wc"]),
            textcoords="offset points",
            xytext=(6, 6),
        )
        axes[0, 1].plot([row["roc_auc"], row["roc_auc_wc"]], [row["pr_auc"], row["pr_auc_wc"]], color="#475569")
    axes[0, 1].set_xlabel("Run-level ROC-AUC")
    axes[0, 1].set_ylabel("Run-level Average Precision")
    axes[0, 1].set_title("B. Digital-twin score refinement")

    x = np.arange(len(summary))
    axes[1, 0].bar(x, summary["benign_block_false_alarm_rate"], color="#f59e0b")
    axes[1, 0].axhline(float(summary["target_alpha"].iloc[0]), color="black", linestyle="--", linewidth=1.2)
    axes[1, 0].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 0].set_ylabel("Empirical benign block FAR")
    axes[1, 0].set_title("C. Calibrated reliability")

    bars = axes[1, 1].bar(x, summary["anomaly_detect_rate"], color="#16a34a", label="Detection rate")
    ax2 = axes[1, 1].twinx()
    ax2.plot(x, summary["median_time_to_detect_s"], color="#1d4ed8", marker="o", linewidth=2, label="Median TTD")
    axes[1, 1].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 1].set_ylabel("Run-level detection rate")
    ax2.set_ylabel("Median time-to-detect (s)")
    axes[1, 1].set_title("D. Operational decision performance")
    axes[1, 1].legend([bars], ["Detection rate"], loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_performance_stack.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return summary, png


def render_explainability_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Path]:
    tier_contrib = pd.read_csv(out_full / "stressor_tier_contributions.csv")
    mechanism = pd.read_csv(out_full / "mechanism_group_summary.csv")
    cm = pd.read_csv(out_full / "stressor_confusion_matrix.csv", index_col=0)

    tier_contrib.to_csv(paper_full / "paper_tier_contribution_summary.csv", index=False)
    mechanism.to_csv(paper_full / "paper_mechanism_summary.csv", index=False)
    cm.to_csv(paper_full / "paper_stressor_confusion_matrix.csv")

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    x = np.arange(len(tier_contrib))
    axes[0].bar(x, tier_contrib["tier0_share"], label="Tier-0")
    axes[0].bar(x, tier_contrib["tier1_alt_share"], bottom=tier_contrib["tier0_share"], label="Tier-1")
    axes[0].bar(
        x,
        tier_contrib["tier2_share"],
        bottom=tier_contrib["tier0_share"] + tier_contrib["tier1_alt_share"],
        label="Tier-2",
    )
    axes[0].set_xticks(x, tier_contrib["stressor"], rotation=30, ha="right")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("A. Tier contribution by stressor")
    axes[0].legend(loc="upper right")

    mech_cols = [
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
    ]
    bottom = np.zeros(len(mechanism))
    colors = ["#0f766e", "#1d4ed8", "#dc2626", "#9333ea", "#b45309"]
    for col, color in zip(mech_cols, colors):
        axes[1].bar(np.arange(len(mechanism)), mechanism[col], bottom=bottom, label=col.replace("_share", ""), color=color)
        bottom += mechanism[col].to_numpy(dtype=float)
    axes[1].set_xticks(np.arange(len(mechanism)), mechanism["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("B. Mechanism evidence by stressor")
    axes[1].legend(loc="upper right", fontsize=7)

    im = axes[2].imshow(cm.values, cmap="Blues")
    axes[2].set_xticks(range(len(cm.columns)), list(cm.columns), rotation=30, ha="right")
    axes[2].set_yticks(range(len(cm.index)), list(cm.index))
    axes[2].set_title("C. Stressor diagnosis confusion")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[2].text(j, i, str(int(cm.iloc[i, j])), ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.tight_layout()
    png = paper_fig / "fig_paper_explainability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_contrib, mechanism, cm, png


def render_portability_dashboard(
    frontier: pd.DataFrame,
    holdout: pd.DataFrame,
    bootstrap_ci: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    view = frontier.copy()
    view["config_label"] = _cfg_labels(view["config"])
    holdout_view = holdout.copy()
    holdout_view["config_label"] = _cfg_labels(holdout_view["config"])
    boot_view = bootstrap_ci.copy()
    boot_view["config_label"] = _cfg_labels(boot_view["config"])

    portability_summary = view[
        [
            "config",
            "config_label",
            "n_features",
            "portable_pr_auc",
            "holdout_worst_pr_auc",
            "reliability_margin",
            "joint_detection_diagnosis",
        ]
    ].copy()
    portability_summary.to_csv(paper_full / "paper_portability_summary.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    scatter = axes[0].scatter(
        view["n_features"],
        view["portable_pr_auc"],
        s=view["joint_detection_diagnosis"].fillna(0.0) * 1800 + 140,
        c=view["reliability_margin"],
        cmap="viridis",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in view.iterrows():
        axes[0].annotate(row["config_label"], (row["n_features"], row["portable_pr_auc"]), textcoords="offset points", xytext=(6, 6))
    axes[0].set_xlabel("Median active features")
    axes[0].set_ylabel("Portable AUC-PR")
    axes[0].set_title("A. Observability-portability frontier")
    fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(holdout_view))
    width = 0.35
    axes[1].bar(x - width / 2.0, holdout_view["mean_pr_auc"], width=width, label="Mean holdout PR")
    axes[1].bar(x + width / 2.0, holdout_view["worst_pr_auc"], width=width, label="Worst holdout PR")
    axes[1].set_xticks(x, holdout_view["config_label"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_title("B. Holdout portability")
    axes[1].legend(loc="upper right")

    x = np.arange(len(boot_view))
    pr_mid = (boot_view["pr_auc_wc_lo"] + boot_view["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - boot_view["pr_auc_wc_lo"], boot_view["pr_auc_wc_hi"] - pr_mid])
    axes[2].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4, color="#1d4ed8", label="AP CI")
    det_mid = (boot_view["detect_lo"] + boot_view["detect_hi"]) / 2.0
    det_err = np.vstack([det_mid - boot_view["detect_lo"], boot_view["detect_hi"] - det_mid])
    axes[2].errorbar(x, det_mid, yerr=det_err, fmt="o", capsize=4, color="#16a34a", label="Detect-rate CI")
    axes[2].set_xticks(x, boot_view["config_label"], rotation=30, ha="right")
    axes[2].set_ylim(0.0, 1.05)
    axes[2].set_title("C. Bootstrap uncertainty")
    axes[2].legend(loc="lower right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_portability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return portability_summary, png


In [ ]:
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## 6. DICE-Specific Design-Space Evaluation

This section turns the completed DICE run into a design-space study centered on the digital twin rather than only a single ROC/PR table.

Sweeps 1, 4, and 5 are derived from the main released outputs. Sweeps 2 and 3 come from the tuning outputs and are available after the notebook is run with `INCLUDE_TUNING = True`.


In [ ]:
display(Markdown("### DICE visual design-space exploration"))

paper_full = OUT_PAPER / "full"
paper_full.mkdir(parents=True, exist_ok=True)

overall_full = pd.read_csv(OUT_FULL / "overall_metrics.csv")
sequential = pd.read_csv(OUT_FULL / "sequential_metrics.csv")
diagnosis = pd.read_csv(OUT_FULL / "stressor_diagnosis_metrics.csv")

holdout_path = OUT_HOLDOUT / "holdout_robustness_summary.csv"
holdout = pd.read_csv(holdout_path) if holdout_path.exists() else None

reliability_path = paper_full / "conformal_reliability_summary.csv"
reliability = pd.read_csv(reliability_path) if reliability_path.exists() else None

stage1_path = DATASET_ROOT / "results_dice_tuning" / "sweep_stage1_gain_block.csv"
stage2_path = DATASET_ROOT / "results_dice_tuning" / "sweep_stage2_alpha_persist.csv"
stage1 = pd.read_csv(stage1_path) if stage1_path.exists() else None
stage2 = pd.read_csv(stage2_path) if stage2_path.exists() else None

config_label_map = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}
feature_budget_map = {
    "tier0": 46,
    "tier0_tier1": 57,
    "tier0_tier1_tier2": 64,
}


def draw_heatmap(ax, pivot_df, title, cmap="viridis", vmin=None, vmax=None, fmt="{:.3f}"):
    vals = pivot_df.values.astype(float)
    im = ax.imshow(vals, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(pivot_df.shape[1]))
    ax.set_xticklabels([str(c) for c in pivot_df.columns])
    ax.set_yticks(np.arange(pivot_df.shape[0]))
    ax.set_yticklabels([str(i) for i in pivot_df.index])
    ax.set_title(title, fontsize=12, fontweight="bold")
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            text = "" if np.isnan(vals[i, j]) else fmt.format(vals[i, j])
            color = "white" if not np.isnan(vals[i, j]) and vals[i, j] > 0.55 else "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=9, color=color)
    return im


def load_final_head_run_metrics(out_dir: Path) -> dict:
    out = {}
    seq_file = out_dir / "sequential_metrics.csv"
    ov_file = out_dir / "overall_metrics.csv"

    if seq_file.exists():
        seq_df = pd.read_csv(seq_file)
        if "tier0_tier1_tier2" in set(seq_df["config"]):
            seq_row = seq_df[seq_df["config"] == "tier0_tier1_tier2"].iloc[0]
            out.update(
                {
                    "benign_run_alert_rate": seq_row.get("benign_run_alert_rate", np.nan),
                    "anomaly_detect_rate": seq_row.get("anomaly_detect_rate", np.nan),
                    "median_time_to_detect_s": seq_row.get("median_time_to_detect_s", np.nan),
                }
            )

    if ov_file.exists():
        ov_df = pd.read_csv(ov_file)
        if "tier0_tier1_tier2" in set(ov_df["config"]):
            ov_row = ov_df[ov_df["config"] == "tier0_tier1_tier2"].iloc[0]
            out.update(
                {
                    "roc_auc_wc": ov_row.get("roc_auc_wc", np.nan),
                    "pr_auc_wc": ov_row.get("pr_auc_wc", np.nan),
                    "fpr_run_alert": ov_row.get("fpr_run_alert", np.nan),
                    "tpr_run_alert": ov_row.get("tpr_run_alert", np.nan),
                }
            )

    return out


def enrich_tuning_table(df: pd.DataFrame | None) -> pd.DataFrame | None:
    if df is None or df.empty:
        return df

    rows = []
    for _, row in df.iterrows():
        rec = row.to_dict()

        out_dir = None
        if "out_dir" in rec and pd.notna(rec["out_dir"]):
            out_dir = Path(rec["out_dir"])

        extra = load_final_head_run_metrics(out_dir) if out_dir and out_dir.exists() else {}

        for key in [
            "roc_auc_wc",
            "pr_auc_wc",
            "fpr_run_alert",
            "tpr_run_alert",
            "benign_run_alert_rate",
            "anomaly_detect_rate",
            "median_time_to_detect_s",
        ]:
            if key not in rec or pd.isna(rec.get(key, np.nan)):
                rec[key] = extra.get(key, np.nan)

        if pd.isna(rec.get("fpr_run_alert", np.nan)):
            rec["fpr_run_alert"] = rec.get("benign_run_alert_rate", np.nan)
        if pd.isna(rec.get("tpr_run_alert", np.nan)):
            rec["tpr_run_alert"] = rec.get("anomaly_detect_rate", np.nan)

        rows.append(rec)

    return pd.DataFrame(rows)


def pareto_front(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    keep = []
    for i, row_i in df.iterrows():
        dominated = False
        xi, yi = row_i[x_col], row_i[y_col]
        for j, row_j in df.iterrows():
            if i == j:
                continue
            xj, yj = row_j[x_col], row_j[y_col]
            if (xj <= xi and yj >= yi) and (xj < xi or yj > yi):
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return df.loc[keep].sort_values([x_col, y_col])


stage1 = enrich_tuning_table(stage1)
stage2 = enrich_tuning_table(stage2)

frontier = (
    overall_full[["config", "roc_auc_wc", "pr_auc_wc"]]
    .merge(
        sequential[
            ["config", "benign_run_alert_rate", "anomaly_detect_rate", "median_time_to_detect_s"]
        ],
        on="config",
        how="left",
    )
    .merge(
        diagnosis[["config", "top1_acc", "top2_acc"]],
        on="config",
        how="left",
    )
)

if holdout is not None:
    frontier = frontier.merge(
        holdout[["config", "mean_pr_auc", "worst_pr_auc"]],
        on="config",
        how="left",
    )
else:
    frontier["mean_pr_auc"] = np.nan
    frontier["worst_pr_auc"] = np.nan

if reliability is not None:
    frontier = frontier.merge(
        reliability[["config", "benign_block_false_alarm_rate", "target_alpha"]],
        on="config",
        how="left",
    )
    frontier["reliability_margin"] = frontier["target_alpha"] - frontier["benign_block_false_alarm_rate"]
else:
    frontier["target_alpha"] = 0.05
    frontier["reliability_margin"] = frontier["target_alpha"] - frontier["benign_run_alert_rate"]

frontier["label"] = frontier["config"].map(config_label_map)
frontier["n_features"] = frontier["config"].map(feature_budget_map)
frontier["portable_pr_auc"] = frontier["mean_pr_auc"].fillna(frontier["pr_auc_wc"])
frontier["holdout_worst_pr_auc"] = frontier["worst_pr_auc"]
frontier["joint_detection_diagnosis"] = frontier["anomaly_detect_rate"] * frontier["top1_acc"]

key_rows = []

sweep1 = frontier[
    [
        "label",
        "n_features",
        "pr_auc_wc",
        "roc_auc_wc",
        "anomaly_detect_rate",
        "benign_run_alert_rate",
        "median_time_to_detect_s",
        "top1_acc",
        "top2_acc",
    ]
].copy().sort_values("n_features")

fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.0))

bubble_sizes = 180 + 900 * sweep1["top2_acc"].fillna(0)
sc = axes[0].scatter(
    sweep1["n_features"],
    sweep1["pr_auc_wc"],
    s=bubble_sizes,
    c=sweep1["anomaly_detect_rate"],
    cmap="viridis",
    edgecolor="black",
    linewidth=1.0,
)
for _, row in sweep1.iterrows():
    axes[0].text(row["n_features"] + 0.8, row["pr_auc_wc"] + 0.01, row["label"], fontsize=10)
axes[0].plot(sweep1["n_features"], sweep1["pr_auc_wc"], linestyle="--", color="#94A3B8", alpha=0.9)
axes[0].set_xlabel("Active feature budget")
axes[0].set_ylabel("Workload-conditioned AUC-PR")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Sweep 1A: observability frontier", fontweight="bold")
axes[0].grid(alpha=0.20)
fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label="Detection rate")

y = np.arange(len(sweep1))
axes[1].hlines(y, sweep1["benign_run_alert_rate"], sweep1["anomaly_detect_rate"], color="#CBD5E1", linewidth=4)
axes[1].scatter(sweep1["benign_run_alert_rate"], y, s=120, color="#F28E2B", label="Benign alert rate", zorder=3)
axes[1].scatter(sweep1["anomaly_detect_rate"], y, s=120, color="#E15759", label="Detection rate", zorder=3)
for idx, row in enumerate(sweep1.itertuples()):
    axes[1].text(max(row.benign_run_alert_rate, row.anomaly_detect_rate) + 0.02, idx, f"TTD={row.median_time_to_detect_s:.0f}s", va="center", fontsize=9)
axes[1].set_yticks(y)
axes[1].set_yticklabels(sweep1["label"])
axes[1].set_xlim(0, 1.05)
axes[1].set_xlabel("Rate")
axes[1].set_title("Sweep 1B: operational separation", fontweight="bold")
axes[1].legend(frameon=False, loc="lower right")
axes[1].grid(axis="x", alpha=0.20)

fig.suptitle("Sweep 1: Observability heads", fontsize=15, fontweight="bold")
fig.tight_layout()
sweep1_png = paper_full / "fig_sweep_1_observability_creative.png"
fig.savefig(sweep1_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(sweep1_png)))

best_head = frontier.sort_values("pr_auc_wc", ascending=False).iloc[0]
key_rows.append(
    {
        "Sweep": "1",
        "Main result": f"Best head: {best_head['label']}",
        "Key metric": f"AUC-PR={best_head['pr_auc_wc']:.3f}, detect={best_head['anomaly_detect_rate']:.3f}",
    }
)

if stage1 is not None and not stage1.empty:
    stage1_ok = stage1[stage1["status"] == "ok"].copy() if "status" in stage1.columns else stage1.copy()
    stage1_ok = stage1_ok.dropna(subset=["pr_auc_wc", "fpr_run_alert", "tpr_run_alert"], how="any").copy()
    stage1_ok["stability_margin"] = stage1_ok["tpr_run_alert"] - stage1_ok["fpr_run_alert"]

    pr_pivot = stage1_ok.pivot_table(index="gain", columns="block_B", values="pr_auc_wc", aggfunc="mean").sort_index().sort_index(axis=1)
    sm_pivot = stage1_ok.pivot_table(index="gain", columns="block_B", values="stability_margin", aggfunc="mean").sort_index().sort_index(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.9))
    im1 = draw_heatmap(axes[0], pr_pivot, "AUC-PR", cmap="Blues", vmin=0, vmax=1)
    im2 = draw_heatmap(axes[1], sm_pivot, "Stability margin (TPR - FPR)", cmap="YlGn", vmin=0, vmax=1)
    axes[0].set_xlabel("Block B")
    axes[0].set_ylabel("Gain")
    axes[1].set_xlabel("Block B")
    axes[1].set_ylabel("Gain")
    fig.suptitle("Sweep 2: Twin synchronization map", fontsize=15, fontweight="bold")
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    fig.tight_layout()
    sweep2_png = paper_full / "fig_sweep_2_gain_block_heatmaps.png"
    fig.savefig(sweep2_png, dpi=220, bbox_inches="tight")
    plt.close(fig)

    display(Image(filename=str(sweep2_png)))

    best_stage1 = stage1_ok.sort_values(["pr_auc_wc", "stability_margin"], ascending=False).iloc[0]
    key_rows.append(
        {
            "Sweep": "2",
            "Main result": f"Best gain/block: g={best_stage1['gain']}, B={int(best_stage1['block_B'])}",
            "Key metric": f"AUC-PR={best_stage1['pr_auc_wc']:.3f}, stability={best_stage1['stability_margin']:.3f}",
        }
    )
else:
    display(Markdown("**Sweep 2 skipped:** tuning outputs not found."))
    key_rows.append(
        {
            "Sweep": "2",
            "Main result": "Skipped",
            "Key metric": "No tuning outputs found",
        }
    )

if stage2 is not None and not stage2.empty:
    stage2_ok = stage2[stage2["status"] == "ok"].copy() if "status" in stage2.columns else stage2.copy()
    stage2_ok = stage2_ok.dropna(subset=["fpr_run_alert", "tpr_run_alert"], how="any").copy()
    stage2_ok["stability_margin"] = stage2_ok["tpr_run_alert"] - stage2_ok["fpr_run_alert"]

    pf = pareto_front(stage2_ok, "fpr_run_alert", "tpr_run_alert")

    fig, ax = plt.subplots(figsize=(9.2, 5.6))
    sizes = 110 + 70 * stage2_ok["persist_k"].astype(float)
    sc = ax.scatter(
        stage2_ok["fpr_run_alert"],
        stage2_ok["tpr_run_alert"],
        s=sizes,
        c=stage2_ok["alpha"],
        cmap="plasma_r",
        edgecolor="black",
        linewidth=0.8,
        alpha=0.90,
    )
    if len(pf) > 1:
        ax.plot(pf["fpr_run_alert"], pf["tpr_run_alert"], linestyle="--", color="black", linewidth=1.7, label="Pareto front")

    for _, row in pf.iterrows():
        ax.annotate(
            f"α={row['alpha']:.2f}, k={int(row['persist_k'])}",
            (row["fpr_run_alert"], row["tpr_run_alert"]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=9,
        )

    ax.set_xlabel("Benign alert rate (lower is better)")
    ax.set_ylabel("Detection rate (higher is better)")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_title("Sweep 3: Decision calibration frontier", fontweight="bold")
    ax.grid(alpha=0.20)
    if len(pf) > 1:
        ax.legend(frameon=False, loc="lower right")

    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="alpha")
    fig.tight_layout()
    sweep3_png = paper_full / "fig_sweep_3_policy_frontier.png"
    fig.savefig(sweep3_png, dpi=220, bbox_inches="tight")
    plt.close(fig)

    display(Image(filename=str(sweep3_png)))

    best_stage2 = stage2_ok.sort_values(["stability_margin", "pr_auc_wc"], ascending=False).iloc[0]
    key_rows.append(
        {
            "Sweep": "3",
            "Main result": f"Best alpha/persist: a={best_stage2['alpha']}, k={int(best_stage2['persist_k'])}",
            "Key metric": f"TPR={best_stage2['tpr_run_alert']:.3f}, FPR={best_stage2['fpr_run_alert']:.3f}",
        }
    )
else:
    display(Markdown("**Sweep 3 skipped:** tuning outputs not found."))
    key_rows.append(
        {
            "Sweep": "3",
            "Main result": "Skipped",
            "Key metric": "No tuning outputs found",
        }
    )

sweep4 = frontier[["label", "n_features", "top1_acc", "top2_acc", "joint_detection_diagnosis"]].copy().sort_values("n_features")

fig, ax = plt.subplots(figsize=(8.8, 5.0))
y = np.arange(len(sweep4))
ax.hlines(y, sweep4["top1_acc"], sweep4["top2_acc"], color="#CBD5E1", linewidth=4)
ax.scatter(sweep4["top1_acc"], y, s=130, color="#4E79A7", label="Top-1 diagnosis", zorder=3)
ax.scatter(sweep4["top2_acc"], y, s=130, color="#59A14F", label="Top-2 diagnosis", zorder=3)
for idx, row in enumerate(sweep4.itertuples()):
    ax.text(row.top2_acc + 0.02, idx, f"{row.label} ({row.n_features} fts)", va="center", fontsize=10)
ax.set_yticks(y)
ax.set_yticklabels([""] * len(y))
ax.set_xlim(0, 1.05)
ax.set_xlabel("Diagnosis accuracy")
ax.set_title("Sweep 4: Diagnosis quality vs feature budget", fontweight="bold")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="x", alpha=0.20)
plt.tight_layout()
sweep4_png = paper_full / "fig_sweep_4_diag_dumbbell.png"
fig.savefig(sweep4_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(sweep4_png)))

best_diag = frontier.sort_values("top2_acc", ascending=False).iloc[0]
key_rows.append(
    {
        "Sweep": "4",
        "Main result": f"Best diagnosis head: {best_diag['label']}",
        "Key metric": f"Top-1={best_diag['top1_acc']:.3f}, Top-2={best_diag['top2_acc']:.3f}",
    }
)

sweep5 = frontier[["label", "n_features", "portable_pr_auc", "holdout_worst_pr_auc", "reliability_margin"]].copy().sort_values("n_features")

fig, ax = plt.subplots(figsize=(8.8, 5.2))
sizes = 120 + 2200 * np.clip(sweep5["reliability_margin"].fillna(0), 0, None)
sc = ax.scatter(
    sweep5["n_features"],
    sweep5["portable_pr_auc"],
    s=sizes,
    c=sweep5["holdout_worst_pr_auc"].fillna(sweep5["portable_pr_auc"]),
    cmap="magma",
    edgecolor="black",
    linewidth=1.0,
    zorder=3,
)

for i in range(len(sweep5) - 1):
    ax.annotate(
        "",
        xy=(sweep5["n_features"].iloc[i + 1], sweep5["portable_pr_auc"].iloc[i + 1]),
        xytext=(sweep5["n_features"].iloc[i], sweep5["portable_pr_auc"].iloc[i]),
        arrowprops=dict(arrowstyle="->", color="#64748B", linewidth=1.6),
    )

for _, row in sweep5.iterrows():
    ax.text(
        row["n_features"] + 0.8,
        row["portable_pr_auc"] + 0.01,
        f"{row['label']}\nworst={row['holdout_worst_pr_auc']:.3f}",
        fontsize=9.5,
        va="center",
    )

ax.set_xlabel("Active feature budget")
ax.set_ylabel("Portable mean AUC-PR")
ax.set_ylim(0, 1.05)
ax.set_title("Sweep 5: Portability frontier", fontweight="bold")
ax.grid(alpha=0.20)

fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="Worst-workload AUC-PR")
plt.tight_layout()
sweep5_png = paper_full / "fig_sweep_5_portability_frontier.png"
fig.savefig(sweep5_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(sweep5_png)))

best_port = frontier.sort_values("portable_pr_auc", ascending=False).iloc[0]
key_rows.append(
    {
        "Sweep": "5",
        "Main result": f"Best portability head: {best_port['label']}",
        "Key metric": f"Portable AUC-PR={best_port['portable_pr_auc']:.3f}",
    }
)

key_df = pd.DataFrame(key_rows)
display(Markdown("### Key DSE takeaways"))
display(key_df)
key_df.to_csv(paper_full / "dse_key_takeaways.csv", index=False)


## 7. Projected DICE-Score Accelerator Complexity

This section estimates only the fixed-point **DICE-score** stage, not the full host-edge software pipeline.

Included in the estimate:
- blockwise accumulation of `|r_{t,j}|`
- weighted reduction into a scalar score
- threshold and persistence logic
- top-k attribution buffer

Excluded from the estimate:
- telemetry ingestion
- normalization
- digital-twin state update / synchronization
- host software runtime

This is therefore a **projected co-design complexity estimate**, not a synthesized die-area or power result.


In [ ]:
display(Markdown("### Projected fixed-point DICE-score accelerator complexity"))

accel_metrics = pd.read_csv(OUT_FULL / "overall_metrics.csv").copy().sort_values("n_features").reset_index(drop=True)
accel_metrics["label"] = accel_metrics["config"].map(CFG_LABEL).fillna(accel_metrics["config"])

# Projected score-stage assumptions only.
BLOCK_B = 60
DECISION_HZ = 1
INPUT_WIDTH_BITS = 16
WEIGHT_WIDTH_BITS = 16
ACC_WIDTH_BITS = 32
SCORE_WIDTH_BITS = 24
TOPK = 5
PERSIST_COUNTER_BITS = 8

rows = []
for _, row in accel_metrics.iterrows():
    n = int(row["n_features"])
    feature_index_bits = max(1, int(np.ceil(np.log2(max(n, 2)))))

    abs_ops_per_sample = n
    accum_adds_per_sample = n
    sample_path_ops_per_block = (abs_ops_per_sample + accum_adds_per_sample) * BLOCK_B

    mult_ops_per_block = n
    reduce_adds_per_block = max(n - 1, 0)
    topk_compare_ops_per_block = n * TOPK
    threshold_compare_ops_per_block = 1
    persist_compare_ops_per_block = 1
    block_end_ops_per_block = (
        mult_ops_per_block
        + reduce_adds_per_block
        + topk_compare_ops_per_block
        + threshold_compare_ops_per_block
        + persist_compare_ops_per_block
    )

    accumulator_state_bits = n * ACC_WIDTH_BITS
    weight_storage_bits = n * WEIGHT_WIDTH_BITS
    topk_state_bits = TOPK * (feature_index_bits + SCORE_WIDTH_BITS)
    control_state_bits = SCORE_WIDTH_BITS + 2 * PERSIST_COUNTER_BITS + 16
    total_state_bits = (
        accumulator_state_bits
        + weight_storage_bits
        + topk_state_bits
        + control_state_bits
    )

    rows.append(
        {
            "config": row["config"],
            "label": row["label"],
            "n_features": n,
            "abs_ops_per_sample": abs_ops_per_sample,
            "accum_adds_per_sample": accum_adds_per_sample,
            "sample_path_ops_per_block": sample_path_ops_per_block,
            "mult_ops_per_block": mult_ops_per_block,
            "reduce_adds_per_block": reduce_adds_per_block,
            "topk_compare_ops_per_block": topk_compare_ops_per_block,
            "block_end_ops_per_block": block_end_ops_per_block,
            "accumulator_state_bits": accumulator_state_bits,
            "weight_storage_bits": weight_storage_bits,
            "topk_state_bits": topk_state_bits,
            "total_state_bits": total_state_bits,
            "total_state_bytes": int(np.ceil(total_state_bits / 8.0)),
        }
    )

accel_df = pd.DataFrame(rows)
base_sample = accel_df["sample_path_ops_per_block"].min()
base_block = accel_df["block_end_ops_per_block"].min()
base_state = accel_df["total_state_bytes"].min()

accel_df["sample_path_norm_vs_tier0"] = accel_df["sample_path_ops_per_block"] / base_sample
accel_df["block_end_norm_vs_tier0"] = accel_df["block_end_ops_per_block"] / base_block
accel_df["state_norm_vs_tier0"] = accel_df["total_state_bytes"] / base_state
accel_df["complexity_index"] = (
    0.40 * accel_df["sample_path_norm_vs_tier0"]
    + 0.35 * accel_df["block_end_norm_vs_tier0"]
    + 0.25 * accel_df["state_norm_vs_tier0"]
)

accel_csv = PAPER_FULL / "projected_dice_score_accelerator_complexity.csv"
accel_df.to_csv(accel_csv, index=False)

display(
    accel_df[
        [
            "label",
            "n_features",
            "abs_ops_per_sample",
            "accum_adds_per_sample",
            "mult_ops_per_block",
            "topk_compare_ops_per_block",
            "block_end_ops_per_block",
            "total_state_bytes",
            "complexity_index",
        ]
    ].round(3)
)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8))
x = np.arange(len(accel_df))

axes[0].bar(x, accel_df["abs_ops_per_sample"], color="#4E79A7", label="abs")
axes[0].bar(
    x,
    accel_df["accum_adds_per_sample"],
    bottom=accel_df["abs_ops_per_sample"],
    color="#59A14F",
    label="accumulate",
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(accel_df["label"])
axes[0].set_ylabel("Ops per 1 Hz sample")
axes[0].set_title("Streaming sample path", fontweight="bold")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.20)

axes[1].bar(x, accel_df["mult_ops_per_block"], color="#E15759", label="weighted mult")
axes[1].bar(
    x,
    accel_df["reduce_adds_per_block"],
    bottom=accel_df["mult_ops_per_block"],
    color="#F28E2B",
    label="score reduction",
)
axes[1].bar(
    x,
    accel_df["topk_compare_ops_per_block"],
    bottom=accel_df["mult_ops_per_block"] + accel_df["reduce_adds_per_block"],
    color="#B07AA1",
    label="top-k compare",
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(accel_df["label"])
axes[1].set_ylabel("Ops per block")
axes[1].set_title("Block-end score path", fontweight="bold")
axes[1].legend(frameon=False)
axes[1].grid(axis="y", alpha=0.20)

axes[2].bar(x, accel_df["total_state_bytes"], color="#76B7B2", label="State bytes")
axes2 = axes[2].twinx()
line = axes2.plot(x, accel_df["complexity_index"], color="black", marker="o", linewidth=2, label="Complexity index")[0]
axes[2].set_xticks(x)
axes[2].set_xticklabels(accel_df["label"])
axes[2].set_ylabel("State bytes")
axes2.set_ylabel("Normalized complexity")
axes[2].set_title("Projected storage footprint", fontweight="bold")
axes[2].grid(axis="y", alpha=0.20)
axes[2].legend([axes[2].patches[0], line], ["State bytes", "Complexity index"], frameon=False, loc="upper left")

fig.suptitle("Projected fixed-point DICE-score accelerator complexity", fontsize=16, fontweight="bold")
fig.tight_layout()

accel_png = PAPER_FIG / "fig_projected_dice_score_accelerator_complexity.png"
fig.savefig(accel_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(accel_png)))

display(
    Markdown(
        f"""
**Assumptions**
- `B = {BLOCK_B}` second decision blocks on a `1 Hz` grid
- residual input width = `{INPUT_WIDTH_BITS}` bits
- weight width = `{WEIGHT_WIDTH_BITS}` bits
- accumulator width = `{ACC_WIDTH_BITS}` bits
- score width = `{SCORE_WIDTH_BITS}` bits
- streaming top-`{TOPK}` attribution
- estimate covers only the fixed-point score stage, not telemetry ingestion or twin synchronization
"""
    )
)


## Next Step

Move to `dice_itc_04_case_study_and_llm.ipynb` for the case study, attribution, and grounded LLM triage section.
